In [0]:
from pyspark.sql.functions import col, when, to_date, date_format
from pyspark.sql.types import DateType

def standardize_date_column(df, column_name):
    return df.withColumn(
        column_name,
        when(
            col(column_name).contains("/"),
            date_format(to_date(col(column_name), "M/d/yyyy"), "yyyy-MM-dd")
        ).when(
            col(column_name).contains("-"),
            date_format(to_date(col(column_name), "MM-dd-yyyy"), "yyyy-MM-dd")
        ).otherwise(
            date_format(to_date(col(column_name), "yyyy-MM-dd"), "yyyy-MM-dd")
        )
    ).withColumn(
        column_name, col(column_name).cast(DateType())
    )

def lowercase_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.lower())
    return df

In [0]:
from pyspark.sql.functions import col, when, lit, to_date, date_format

df = spark.table("01_prod_bronze.raw.stores")
df=standardize_date_column(df,"Open_Date")

# df = df.withColumn("Open_Date", to_date(col("Open_Date"), "yyyy-MM-dd"))

                 
df=lowercase_columns(df)
# display(df)
null_count_filter = df.filter(col("square_meters").isNull()).count()

df=df.fillna({"square_meters":0})
display(df)
# 1. Check for missing/null values in Square_Meters and Open_Date

In [0]:
from pyspark.sql.functions import col, when, to_date, date_format,upper
from pyspark.sql.types import DateType

df = spark.table("01_prod_bronze.raw.sales")
df=standardize_date_column(df,"Delivery_Date")
# df = df.withColumn(
#     "Delivery_Date",
#     when(
#         col("Delivery_Date").contains("/"),
#         date_format(to_date(col("Delivery_Date"), "M/d/yyyy"), "yyyy-MM-dd")
#     ).when(
#         col("Delivery_Date").contains("-"),
#         date_format(to_date(col("Delivery_Date"), "MM-dd-yyyy"), "yyyy-MM-dd")
#     ).otherwise(None)
# )
# df = df.withColumn("Delivery_Date", col("Delivery_Date").cast(DateType()))

# df = df.withColumn(
#     "Order_Date",
#     when(
#         col("Order_Date").contains("/"),
#         date_format(to_date(col("Order_Date"), "M/d/yyyy"), "yyyy-MM-dd")
#     ).when(
#         col("Order_Date").contains("-"),
#         date_format(to_date(col("Order_Date"), "MM-dd-yyyy"), "yyyy-MM-dd")
#     ).otherwise(None)
# )
# df = df.withColumn("Order_Date", col("Order_Date").cast(DateType()))

df=standardize_date_column(df,"Order_Date")
df = df.withColumn(
    "Currency_Code",
    upper(col("currency_code"))
)

df = df.withColumn(
    "Currency_Code",
    when(col("Currency_Code").isin("USD", "US$", "US DOLLAR"), "USD")
    .when(col("Currency_Code").isin("EUR", "EURO"), "EUR")
    .when(col("Currency_Code").isin("GBP", "POUND"), "GBP")
    .when(col("Currency_Code").isin("CAD", "CANADIAN DOLLAR"), "CAD")
    .when(col("Currency_Code").isin("AUD", "AUSTRALIAN DOLLAR"), "AUD")
    .otherwise(col("Currency_Code"))
)

for c in df.columns:
    df = df.withColumnRenamed(c, c.lower())

display(df)

In [0]:
from pyspark.sql.functions import col, when, to_date, date_format

df = spark.table("01_prod_bronze.raw.customers")

# Standardize column names to lowercase
for c in df.columns:
    df = df.withColumnRenamed(c, c.lower())

# Transform birthday column with proper when/otherwise logic
# df = df.withColumn(
#     "birthday",
#     when(
#         col("birthday").contains("/"),
#         date_format(to_date(col("birthday"), "M/d/yyyy"), "yyyy-MM-dd")
#     ).when(
#         col("birthday").contains("-"),
#         date_format(to_date(col("birthday"), "MM-dd-yyyy"), "yyyy-MM-dd")
#     ).otherwise(
#         date_format(to_date(col("birthday"), "yyyy-MM-dd"), "yyyy-MM-dd")
#     )
# )

df=standardize_date_column(df,"birthday")
df = df.withColumn("birthday", col("birthday").cast(DateType()))

# df=df.select(["state_code"]).distinct()
display(df)